# RAG Ecosystem

[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/downloads/release/python-3100/) [![LangChain](https://img.shields.io/badge/LangChain-%23007ACC.svg?logo=LangChain)](https://www.langchain.com/) [![DeepEval](https://img.shields.io/badge/DeepEval-Evaluation-orange)](https://github.com/confident-ai/deepeval) [![RAGAS](https://img.shields.io/badge/RAGAS-Evaluation-blueviolet)](https://github.com/explodinggradients/ragas) [![OpenAI](https://img.shields.io/badge/OpenAI-API-lightgrey)](https://openai.com/) [![Cohere](https://img.shields.io/badge/Cohere-API-yellowgreen)](https://cohere.com/) [![Medium](https://img.shields.io/badge/Medium-Blog-black?logo=medium)](https://medium.com/@fareedkhandev/8f23349b96a4)

Hello 👋🏻 I'm Yashh and in this notebook I've built and implemented a complete RAG Based AI System with optimizing each component. 

 I've made this notebook in such a way that it becomes easy for me to revise and resuse my work, and also for the easy understanding for the viewers.

![image.png](attachment:image.png)

> The Advanced components that we'll be working on in detail : 

- **Query Transformations:** Rewriting user questions to be more effective for retrieval.
- **Intelligent Routing:** Directing a query to the correct data source or a specialized tool.
- **Indexing:** Creating a multi-layered knowledge base.
- **Retrieval and Re-ranking:** Filtering noise and prioritizing the most relevant context.
- **Self-Correcting Agentic Flows:** Building systems that can grade and improve their own work.
- **End-to-End Evaluation:** Objectively measuring the performance of the entire pipeline.

and much more …

> Let's create the RAG!

## Table of Contents

- [Understanding Basic RAG System](#part1)
  - [Indexing Phase](#part1-1)
  - [Retrieval](#part1-2)
  - [Generation](#part1-3)
- [Advanced Query Transformations](#part2)
  - [Multi-Query Generation](#part2-1)
  - [RAG-Fusion](#part2-2)
  - [Decomposition](#part2-3)
  - [Step-Back Prompting](#part2-4)
  - [HyDE](#part2-5)
- [Routing & Query Construction](#part3)
  - [Logical Routing](#part3-1)
  - [Semantic Routing](#part3-2)
  - [Query Structuring](#part3-3)
- [Advanced Indexing Strategies](#part4)
  - [Multi-Representation Indexing](#part4-1)
  - [Hierarchical Indexing (RAPTOR) Knowledge Tree](#part4-2)
  - [Token-Level Precision (ColBERT)](#part4-3)
- [Advanced Retrieval & Generation](#part5)
  - [Dedicated Re-ranking](#part5-1)
  - [Self-Correction using AI Agents](#part5-2)
  - [Impact of Long Context](#part5-3)
- [Manual RAG Evaluation](#part6)
  - [The Core Metrics: What Should We Measure?](#part6-1)
  - [Building Evaluators from Scratch with LangChain](#part6-2)
- [Evaluation with Frameworks](#part7)
  - [Rapid Evaluation with deepeval](#part7-1)
  - [Another Powerful Alternative with grouse](#part7-2)
  - [Evaluation with RAGAS](#part7-3)
- [Summarizing our Work](#part8)

> small caveat to know, we'll be using python 3.10 for this project specifically. I have noticed unusual errors because my system is running on python 3.13. 

> Python 3.10 is stable and supports wider tools that we'll be using for this project

> if your system is running on 3.11+ too, then we explicitly install 3.10 for this project and here are the 3 commands to run. 

1. rm -rf venv (if already made)
2. python3.10 -m venv venv
3. source venv/bin/activate (for mac)
4. source venv/Scripts/activate (for windows)

<a id='part1'></a>
# Basics of a RAG System

Before we get into anything, let’s install core Python libraries commonly used for AI products, such as LangChain and others.

In [1]:
!pip install langchain langchain_community langchain-openai langchainhub chromadb tiktoken

We now set the environment variables for tracing and other tasks, such as the LLMs API provider we will be using.

In [2]:
import os
from dotenv import load_dotenv

# Load variables from .env file
load_dotenv()

os.environ["LANGCHAIN_ENDPOINT"] = os.getenv("LANGCHAIN_ENDPOINT")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

You can obtain your `LangSmith` API key from [their official documentation](https://www.langchain.com/langsmith) to trace our RAG product throughout this blog. For the LLM, we will be using the `Groq` API but as you may already know, `LangChain` supports a variety of LLM providers as well.

Although i'm not using it, `LANGCHAIN_PROJECT` is optional but strongly recommended—it organizes traces in LangSmith.

The core RAG pipeline is the foundation of any advanced system, and understanding its components is important. Therefore, before going into the details of advanced components, we first need to understand the core logic of how a RAG system works, **but you can skip this section if you are already aware of how RAG system works.**


This simplest RAG can be break into three components:

- **Indexing**: Organize and store data in a structured format to enable efficient searching.
- **Retrieval**: Search and fetch relevant data based on a query or input.
- **Generation**: Create a final response or output using the retrieved data.

Let’s build this simple pipeline from the ground up to see how each piece works.

<a id='part1-1'></a>
## Indexing Phase

Before our RAG system can answer any questions, it needs knowledge to draw from. For this, we’ll use a `WebBaseLoader` to pull content directly from [Lilian Weng's excellent blog post](https://lilianweng.github.io/posts/2023-06-23-agent/) on LLM-powered agents.


In [3]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Initialize a web document loader with specific parsing instructions
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),  # URL of the blog post to load
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")  # Only parse specified HTML classes
        )
    ),
)

# Load the filtered content from the web page into documents
docs = loader.load()

/Users/yashhmahajan/Desktop/Comprehensive RAG Ecosystem/rageco/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


The `bs_kwargs` argument helps us target only the relevant HTML tags (`post-content`, `post-title`, etc.), cleaning up our data from the start.

Now that we have the document, we face our first challenge. Feeding a massive document directly into an LLM is inefficient and often impossible due to context window limits.

> This is why **chunking** is a critical step. We need to break the document into smaller, semantically meaningful pieces.

The `RecursiveCharacterTextSplitter` is the recommended tool for this job because it intelligently tries to keep paragraphs and sentences intact.

In [4]:
from langchain_text_splitters.character import RecursiveCharacterTextSplitter

# Create a text splitter to divide text into chunks of 1000 characters with 200-character overlap
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# Split the loaded documents into smaller chunks
splits = text_splitter.split_documents(docs)
print(f"Created {len(splits)} document chunks")

Created 63 document chunks


With `chunk_size=1000`, we are creating chunks of 1000 characters, and `chunk_overlap=200` ensures there is some continuity between them, which helps preserve context.

Our text is now split, but it’s still just text. To perform similarity searches, we need to convert these chunks into numerical representations called **embeddings**. We will then store these embeddings in a **vector store**, which is a specialized database designed for efficient searching of vectors.

* The `Chroma` vector store and `HuggingFaceEmbeddings` make this incredibly simple. 

Just like an LLM API Key, we also need to get API key for embeddings and we have muliple embedding models to choose from. 

If you plan on using this project for commerical usage, you might want to use paid embedding models from OpenAI or others.

But for now, we'll be using a free & open source model.

* Major advantage is huggingfaceembeddings are open source and can be used without any API keys. Also it is free! 

The following line handles both embedding and indexing in one go.

In [5]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# Prepend retrieval instruction to each document (BGE optimization)
splits = [
    Document(
        page_content="Represent this document for retrieval: " + doc.page_content,
        metadata=doc.metadata
    )
    for doc in splits
]

embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "mps"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 4
    }
)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embedding
)

/var/folders/t1/wt965jp13hb3gls9w5dcy1bm0000gn/T/ipykernel_39802/3145315280.py:14: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5731.26it/s]


I have a Macbook M2 8GB (as i make this project) and if you beleive your pc has more ram and cpu power, you can go for bigger and better models like `BAAI/bge-large-en-v1.5` or `BAAI/bge-m3`

* `BAAI/bge-small-en-v1.5` - uses 200MB of VRAM
* `BAAI/bge-base-en-v1.5` - uses 600MB of VRAM
* `BAAI/bge-large-en-v1.5` - uses 1.8GB of VRAM
* `BAAI/bge-m3` - uses 2.2BG of VRAM (only choose when working with multilingual content)

### With our knowledge indexed, we are now ready to start asking questions.

<a id='part1-2'></a>
## Retrieval

The vector store is our library, and the **retriever** is our smart librarian. It takes a user’s query, embeds it, and then fetches the most semantically similar chunks from the vector store.


First, we create a retreiver from a vector store

In [6]:
retriever = vectorstore.as_retriever()

Let’s test it. We’ll ask a question and see what our retriever finds.

In [7]:
# Retrieve relevant documents for a query
docs = retriever.invoke("What is Task Decomposition?")

# Print the content of the first retrieved document
print(docs[0].page_content)

Represent this document for retrieval: Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.
Tree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-first search) with each state evaluated by a classifier (via a prompt) or majority vote.


As you can see, the retriever successfully pulled the most relevant chunk from the blog post that directly discusses “Task decomposition.” This piece of context is exactly what the LLM needs to form an accurate answer.

<a id='part1-3'></a>
## Generation

We have our context, but we need an LLM to read it and formulate a human-friendly answer. This is the **“Generation”** step in RAG.


We'll be using LangChain Hub for the templates.

> For those who dont know, the Hub is a centralized platform for sharing, managing, and collaborating on LangChain artifacts, most importantly for now, prompt templates.

> It is integrated directly into LangSmith and serves as a "GitHub for prompts

you can find it here - https://smith.langchain.com/hub?organizationId=63336e7b-fed5-451d-acf3-8ac3fe222056

First, we need a good prompt template. This instructs the LLM on how to behave. Instead of writing our own, we can pull a pre-optimized one from LangChain Hub.

In [8]:
from langchainhub import Client
client = Client()
prompt = client.pull("rlm/rag-prompt")
print(prompt)

/var/folders/t1/wt965jp13hb3gls9w5dcy1bm0000gn/T/ipykernel_39802/2354009971.py:3: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt = client.pull("rlm/rag-prompt")
/Users/yashhmahajan/Desktop/Comprehensive RAG Ecosystem/rageco/lib/python3.10/site-packages/langchainhub/client.py:326: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = self.pull_repo(owner_repo_commit)


{"id": ["langchain", "prompts", "chat", "ChatPromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"messages": [{"id": ["langchain", "prompts", "chat", "HumanMessagePromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"prompt": {"id": ["langchain", "prompts", "prompt", "PromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"template": "You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:", "input_variables": ["question", "context"], "template_format": "f-string"}}}}], "input_variables": ["question", "context"]}}


Next, we initialize our LLM

In [9]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

Now for the final step: chaining everything together. Using the LangChain Expression Language (LCEL), we can pipe the output of one component into the input of the next.

In [10]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate

retriever = vectorstore.as_retriever()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

format_docs_runnable = RunnableLambda(format_docs)

prompt = ChatPromptTemplate.from_template(
    "Answer the question based only on the context below:\n\n{context}\n\nQuestion: {question}"
)

rag_chain = (
    {
        "context": retriever | format_docs_runnable,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

Let’s break down this chain:

1. `{"context": retriever | format_docs_runnable, "question": RunnablePassthrough()}`: This part runs in parallel. It sends the user's question to the `retriever` to get documents, which are then formatted into a single string by `format_docs`. Simultaneously, `RunnablePassthrough` passes the original question through unchanged.
2. `| prompt`: The context and question are fed into our prompt template.
3. `| llm`: The formatted prompt is sent to the LLM.
4. `| StrOutputParser()`: This cleans up the LLM's output into a simple string.

Now, let’s invoke the entire chain.

In [11]:
response = rag_chain.invoke("What is Task Decomposition?")
print(response)

Task decomposition is the process of breaking down a complicated task into smaller and simpler steps. This can be done in several ways, including: 

1. Using large language models (LLMs) with simple prompting, 
2. Using task-specific instructions, 
3. With human inputs, 
4. Utilizing techniques like Chain of Thought (CoT) which instructs the model to "think step by step", 
5. Tree of Thoughts which explores multiple reasoning possibilities at each step, and 
6. LLM+P, which involves relying on an external classical planner to do long-horizon planning using the Planning Domain Definition Language (PDDL). 

The goal of task decomposition is to transform big tasks into multiple manageable tasks, making it easier to understand and complete the task.


And there we have it, our RAG pipeline successfully retrieved relevant information about **“Task Decomposition”** and used it to generate a concise, accurate answer. This simple chain forms the foundation upon which we will build more advanced and powerful capabilities.

<a id='part2'></a>
# Advanced Query Transformations

So, now that we understand the fundamentals of RAG pipeline, honestly, production systems often reveal the limitations of this basic approach. One of the most common failure points is the user’s query itself.

![image.png](attachment:image.png)
reference - https://www.langchain.com/blog/query-transformations


* A query might be too specific, too broad, or use different vocabulary than our source documents, leading to poor retrieval results.

* The solution isn’t to blame the user, it’s to make our system smarter. **Query Transformation** is a set of powerful techniques designed to re-write, expand, or break down the original question to significantly improve retrieval accuracy.


Instead of relying on a single query, we’ll engineer multiple, better-informed queries to cast a wider and more accurate net.

To test these new techniques, we will use the same indexed knowledge base from Basic RAG pipeline section that we have just gone through previously. This ensures we can directly compare the results and see the improvements.

As a quick refresher, we need to setup our retreiver and here's how we do it:

In [12]:
# Load the blog post
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

# Split the documents into chunks
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, 
    chunk_overlap=50
)
splits = text_splitter.split_documents(blog_docs)

# Index the chunks in a Chroma vector store
vectorstore = Chroma.from_documents(documents=splits, 
                                    embedding=HuggingFaceEmbeddings())

# Create our retriever
retriever = vectorstore.as_retriever()

/var/folders/t1/wt965jp13hb3gls9w5dcy1bm0000gn/T/ipykernel_39802/1815399923.py:21: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding=HuggingFaceEmbeddings())
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7111.41it/s]


#### Now, with our retriever ready, let’s explore our first query transformation technique.

<a id='part2-1'></a>
## Multi-Query Generation

A single user query represents just one perspective. Distance-based similarity search might miss relevant documents that use synonyms or discuss related concepts.

The Multi-Query approach tackles this by using an LLM to generate several different versions of the user’s question, effectively searching from multiple angles.

![image.png](attachment:image.png)

We’ll start by creating a prompt that instructs the LLM to generate these alternative questions.

##### To understand how it works, 
Let's ask "Who is Yashh Mahajan?" (represents a simple question which will have a simple answer like "Yashh Mahajan is a student")

##### The moment we apply multi query, the LLM is going to generate more questions

* What is Yashh currently doing?
* Who are Yashh's friends?
* What does Yashh likes to do in his free time?

ultimately, by asking & answering more questions, we're getting more information. And that's exactly what we want.

We’ll start by creating a prompt that instructs the LLM to generate these alternative questions.

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Prompt for generating multiple queries
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Original question: {question}"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

# Chain to generate the queries
generate_queries = (
    prompt_perspectives 
    | llm  # Use the already defined ChatGroq instance
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

Let’s test this chain and see what kind of queries it generates for our question. We shall ask related to our blogpost that we took.

In [14]:
question = "What is task decomposition for LLM agents?"
generated_queries_list = generate_queries.invoke({"question": question})

# Filter out empty queries
cleaned_queries = [q.strip() for q in generated_queries_list if q.strip()]

# Print with correct indexing
for i, q in enumerate(cleaned_queries, 1):
    print(f"{i}. {q}")

1. What is the process of breaking down tasks for large language model agents to improve their performance and efficiency?
2. How do LLM agents utilize task decomposition to enhance their ability to understand and complete complex tasks?
3. What role does task decomposition play in the development and training of large language model agents, and what benefits does it provide?
4. Can you explain the concept of task decomposition in the context of LLM agents, including its definition, importance, and applications?
5. How do researchers and developers use task decomposition to design and optimize the architecture of large language model agents, and what are the potential outcomes of this approach?


Amazing Response! The LLM has rephrased our original question using different keywords like “break down complex tasks”, “utilize”, “process” and importantly (in the 5th question) asking 2 questions simultaneously just like a human would. Now, we can retrieve documents for all of these queries and combine the results. A simple way to combine them is to take the unique set of all retrieved documents.

In [15]:
from langchain_core.load import dumps, loads

def get_unique_union(documents: list[list]):
    """ A simple function to get the unique union of retrieved documents """
    # Flatten the list of lists and convert each Document to a string for uniqueness
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    unique_docs = list(set(flattened_docs))
    return [loads(doc) for doc in unique_docs]

# Build the retrieval chain
retrieval_chain = generate_queries | retriever.map() | get_unique_union

# Invoke the chain and check the number of documents retrieved
docs = retrieval_chain.invoke({"question": question})
print(f"Total unique documents retrieved: {len(docs)}")

Total unique documents retrieved: 10


/var/folders/t1/wt965jp13hb3gls9w5dcy1bm0000gn/T/ipykernel_39802/1646526512.py:8: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]


##### Going great!
By searching with five different queries, we retrieved a total of 10 unique documents, likely capturing a more comprehensive set of information than a single query would have. Now we can feed this context into our final RAG chain.

In [16]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

# The final RAG chain
template = """Answer the following question based on this context:

{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
llm 

final_rag_chain = (
    {"context": retrieval_chain, "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

print(final_rag_chain.invoke({"question": question}))

Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down complex tasks into smaller, more manageable subgoals or steps. This is a crucial component of planning in LLM-powered autonomous agents, as it enables the agent to efficiently handle complex tasks by dividing them into simpler, more manageable parts.

Task decomposition can be achieved through various methods, including:

1. Chain of thought (CoT) prompting: This involves instructing the model to "think step by step" to utilize more test-time computation to decompose hard tasks into smaller and simpler steps.
2. Tree of Thoughts: This extends CoT by exploring multiple reasoning possibilities at each step, creating a tree structure.
3. Simple prompting: Using prompts like "Steps for XYZ.\\n1.", "What are the subgoals for achieving XYZ?" to decompose tasks.
4. Task-specific instructions: Using specific instructions like "Write a story outline." for writing a novel.
5. Human inputs: Allowing hu

* This answer is more robust because it’s based on a wider pool of relevant documents.

<a id='part2-2'></a>
## RAG-Fusion

Multi-Query is a great start, but simply taking a union of documents treats them all equally. What if one document was ranked highly by three of our queries, while another was a low-ranked result from only one?

![image.png](attachment:image.png)

The first is clearly more important. RAG-Fusion improves on Multi-Query by not just fetching documents, but also …

> **re-ranking** them using a technique called **Reciprocal Rank Fusion (RRF)**.

RRF intelligently combines results from multiple searches. It boosts the score of documents that appear consistently high across different result lists, pushing the most relevant content to the top.

The code is very similar, but we’ll swap our `get_unique_union` function with an RRF implementation.

In [18]:
def reciprocal_rank_fusion(results: list[list], k=60):
    """ Reciprocal Rank Fusion that intelligently combines multiple ranked lists """
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # The core of RRF: documents ranked higher (lower rank value) get a larger score
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort documents by their new fused scores in descending order
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    return reranked_results

The above function will re-rank the documents after they are fetched through similarity search, but we haven’t initialized it yet. 

> Let's do it in the next step

In [19]:
# Use a slightly different prompt for RAG-Fusion
template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

generate_queries = (
    prompt_rag_fusion 
    | llm
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

# Build the new retrieval chain with RRF
retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rag_fusion.invoke({"question": question})

print(f"Total re-ranked documents retrieved: {len(docs)}")

Total re-ranked documents retrieved: 11


> now we have our documents ranked according to priority

The final chain remains the same, but now it receives a more intelligently ranked context. RAG-Fusion is a powerful, low-effort way to increase the quality of your retrieval and is practiced widely in industry.

<a id='part2-3'></a>
## Decomposition

Some questions are too complex to be answered in a single step. For example, **“What are the main components of an LLM-powered agent, and how do they interact?”** These are actually two questions in one.

![image.png](attachment:image.png)

The Decomposition technique uses an LLM to break down a complex query into a set of simpler, self-contained sub-questions. We can then answer each one and synthesize a final answer.

We’ll start with a prompt designed for this purpose.

In [20]:
# Decomposition prompt
template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Output (3 queries):"""
prompt_decomposition = ChatPromptTemplate.from_template(template)

# Chain to generate sub-questions
generate_queries_decomposition = (
    prompt_decomposition 
    | llm
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

# Generate and print the sub-questions
question = "What are the main components of an LLM-powered autonomous agent system?"
sub_questions = generate_queries_decomposition.invoke({"question": question})
print(sub_questions)

['To break down the question into manageable sub-questions, here are three search queries:', '', '1. **What are the key architectural components of an LLM (Large Language Model)?**', '   - This query focuses on understanding the fundamental parts of a Large Language Model, which is a crucial element of an LLM-powered autonomous agent system.', '', '2. **How do perception and action systems integrate with LLMs in autonomous agents?**', '   - This search query explores how an autonomous agent perceives its environment and takes actions based on the information processed by the LLM, highlighting the interaction between the LLM and other system components.', '', '3. **What role does decision-making and planning play in LLM-powered autonomous agent systems?**', '   - This query delves into the decision-making processes and planning mechanisms within an autonomous agent system that utilizes an LLM, aiming to understand how the system makes informed decisions based on the data and knowledge i

The LLM successfully decomposed our complex question. Now, we can answer each of these individually and combine the results. One effective method is to answer each sub-question and use the resulting Q&A pairs as context to synthesize a final, comprehensive answer.

In [21]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchainhub import Client

client = Client()
prompt_rag = client.pull("rlm/rag-prompt")

def make_safe_prompt(prompt_obj):
    if hasattr(prompt_obj, "invoke"):
        return prompt_obj

    template_text = getattr(prompt_obj, "template", str(prompt_obj))
    safe_template = template_text.replace("{", "{{").replace("}", "}}")

    for var in ("context", "question"):
        safe_template = safe_template.replace(f"{{{{{var}}}}}", f"{{{var}}}")

    return PromptTemplate.from_template(safe_template)

prompt_rag = make_safe_prompt(prompt_rag)

rag_chain = prompt_rag | llm | StrOutputParser()

rag_results = []
for sub_question in sub_questions:
    retrieved_docs = retriever.invoke(sub_question)
    context_docs = "\n\n".join(doc.page_content for doc in retrieved_docs)

    answer = rag_chain.invoke({
        "context": context_docs,
        "question": sub_question
    })
    rag_results.append(answer)

def format_qa_pairs(questions, answers):
    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start=1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
    return formatted_string.strip()

context = format_qa_pairs(sub_questions, rag_results)

template = """Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the original question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = prompt | llm | StrOutputParser()

print(final_rag_chain.invoke({
    "context": context,
    "question": question
}))

/var/folders/t1/wt965jp13hb3gls9w5dcy1bm0000gn/T/ipykernel_39802/4223813121.py:6: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt_rag = client.pull("rlm/rag-prompt")
/Users/yashhmahajan/Desktop/Comprehensive RAG Ecosystem/rageco/lib/python3.10/site-packages/langchainhub/client.py:326: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = self.pull_repo(owner_repo_commit)


The main components of an LLM-powered autonomous agent system include the Large Language Model (LLM), planning, memory, and a natural language interface. The LLM functions as the agent's brain, generating text and making decisions, while the planning component breaks down large tasks into smaller subgoals and refines plans. The memory component stores and retrieves information, and the natural language interface parses and generates human-readable text, enabling the agent to interact with its environment and make informed decisions based on the data and knowledge it has.


The main components of an LLM-powered autonomous agent system include the Large Language Model (LLM), planning, memory, and a natural language interface. The LLM functions as the agent's brain, generating text and making decisions, while the planning component breaks down large tasks into smaller subgoals and refines plans. The memory component stores and retrieves information, and the natural language interface parses and generates human-readable text, enabling the agent to interact with its environment and make informed decisions based on the data and knowledge it has.

> This is the answer we got in the above cell, and hence, By breaking the problem down, we constructed a much more detailed and structured answer than we would have otherwise.

<a id='part2-4'></a>
## Step-Back Prompting

Sometimes, a user’s query is too specific, while our documents contain the more general, underlying information needed to answer it.

![image.png](attachment:image.png)

> For example, a user might ask, “Could the members of The Police perform lawful arrests?”

A direct search for this might fail. The Step-Back technique uses an LLM to take a “step back” and form a more general question, like “What are the powers and duties of the band The Police?” We then retrieve context for *both* the specific and general questions, providing a richer context for the final answer.

We can teach the LLM this pattern using few-shot examples.

By breaking the problem down, we constructed a much more detailed and structured answer than we would have otherwise.

In [22]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# Few-shot examples to teach the model how to generate step-back (more generic) questions
examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel's was born in what country?",
        "output": "what is Jan Sindel's personal history?",
    },
]

# Define how each example is formatted in the prompt
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),  # User input
    ("ai", "{output}")     # Model's response
])

# Wrap the few-shot examples into a reusable prompt template
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# Full prompt includes system instruction, few-shot examples, and the user question
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "You are an expert at world knowledge. Your task is to step back and paraphrase a question "
     "to a more generic step-back question, which is easier to answer. Here are a few examples:"),
    few_shot_prompt,
    ("user", "{question}"),
])

Now, we can define the chain for step back approach.

In [23]:
generate_queries_step_back = prompt | llm | StrOutputParser()

# Run the chain on a specific question
question = "What is task decomposition for LLM agents?"
step_back_question = generate_queries_step_back.invoke({"question": question})

# Output the original and generated step-back question
print(f"Original Question: {question}")
print(f"Step-Back Question: {step_back_question}")

Original Question: What is task decomposition for LLM agents?
Step-Back Question: What is task decomposition in general?


This is an important step-back question. It broadens the scope to general software engineering, which will likely pull in foundational documents that can then be combined with the specific context about LLM agents. Now we can build a chain that uses both.

In [24]:
from langchain_core.runnables import RunnableLambda

# Prompt for the final response
response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

# Normal Context
{normal_context}

# Step-Back Context
{step_back_context}

# Original Question: {question}
# Answer:"""
response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

# The full chain
chain = (
    {
        # Retrieve context using the normal question
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
        # Retrieve context using the step-back question
        "step_back_context": generate_queries_step_back | retriever,
        # Pass on the original question
        "question": lambda x: x["question"],
    }
    | response_prompt
    | llm 
    | StrOutputParser()
)

response = chain.invoke({"question": question})

In [25]:
print(response)

Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down complex tasks into smaller, more manageable subgoals or steps. This is a crucial component of planning in LLM-powered autonomous agent systems, as it enables the agent to efficiently handle complex tasks.

There are several ways to achieve task decomposition, including:

1. **Chain of Thought (CoT)**: This involves instructing the model to "think step by step" to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and provides insight into the model's thinking process.
2. **Tree of Thoughts**: This extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be performed using breadth-first search (BFS) or depth-first search (DFS) with each state evaluated by a classifier or 

Impressive answer!

<a id='part2-5'></a>
## HyDE

This final technique is one of the most clever. The core problem of retrieval is that a user’s query might use different words than the document (the “vocabulary mismatch” problem).

![image.png](attachment:image.png)

**HyDE (Hypothetical Document Embeddings)** proposes a radical solution: First, have an LLM generate a *hypothetical* answer to the question. This fake document, while not factually correct, will be semantically rich and use the kind of language we expect to find in a real answer.

We then embed this hypothetical document and use its embedding to perform the retrieval. The result is that we find real documents that are semantically very similar to an ideal answer.

Let’s start by creating a prompt to generate this hypothetical document.

In [26]:
# HyDE prompt
template = """Please write a scientific paper passage to answer the question
Question: {question}
Passage:"""
prompt_hyde = ChatPromptTemplate.from_template(template)

# Chain to generate the hypothetical document
generate_docs_for_retrieval = (
    prompt_hyde 
    | llm 
    | StrOutputParser() 
)

# Generate and print the hypothetical document
hypothetical_document = generate_docs_for_retrieval.invoke({"question": question})
print(hypothetical_document)

Task decomposition for Large Language Model (LLM) agents refers to the process of breaking down complex tasks into a series of simpler, more manageable sub-tasks that can be executed by the LLM agent. This technique is essential for enabling LLM agents to tackle complex, high-level tasks that require multiple steps, reasoning, and decision-making.

In task decomposition, the complex task is first analyzed and divided into a set of sub-tasks, each with its own specific objectives and requirements. These sub-tasks are then further decomposed into smaller, more atomic tasks that can be executed by the LLM agent. The LLM agent can then process each sub-task individually, using its language understanding and generation capabilities to produce the required output.

Task decomposition can be performed using various techniques, including hierarchical task decomposition, where tasks are decomposed into sub-tasks in a hierarchical manner, and sequential task decomposition, where tasks are decomp

A typical textbook-styled answer. Now, we use it's embedding to find real documents.

In [27]:
# Retrieve documents using the HyDE approach
retrieval_chain = generate_docs_for_retrieval | retriever 
retrieved_docs = retrieval_chain.invoke({"question": question})

# Use our standard RAG chain to generate the final answer from the retrieved context
response = final_rag_chain.invoke({"context": retrieved_docs, "question": question})
print(response)

Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down complex tasks into smaller, manageable subgoals. This is a crucial component of LLM-powered autonomous agents, as it enables the agent to efficiently handle complex tasks.

There are several techniques used for task decomposition in LLM agents, including:

1. Chain of Thought (CoT): This involves instructing the model to "think step by step" to utilize more test-time computation to decompose hard tasks into smaller and simpler steps.
2. Tree of Thoughts: This extends CoT by exploring multiple reasoning possibilities at each step, creating a tree structure.
3. Simple prompting: LLMs can be prompted with simple questions like "Steps for XYZ" or "What are the subgoals for achieving XYZ?" to decompose tasks.
4. Task-specific instructions: LLMs can be given task-specific instructions, such as "Write a story outline" for writing a novel.
5. Human inputs: Humans can provide input to help the LLM ag

By using a hypothetical document as a **lure**, HyDE helped us zero in on the most relevant chunks in our knowledge base, demonstrating another powerful tool in our RAG toolkit.

<a id='part3'></a>
# Routing & Query Construction

Our RAG system is getting smarter, but in a real-world scenario, knowledge isn’t stored in a single, uniform library.

As you can see in the diagram, this is the next crucial step that comes after Query Transformation.

* Routing directs the quesries to most relevant data source, tools or retreival method, enhancing the accuracy, based on the user intent.

> We often have multiple data sources: documentation for different programming languages, internal wikis, public websites, or databases with structured metadata.

![image.png](attachment:image.png)
reference - https://www.langchain.com/blog/deconstructing-rag#challenge

Sending every query to every source is wildly inefficient and can lead to noisy, irrelevant results.

This is where our RAG system needs to evolve from a simple librarian into an **intelligent switchboard operator**. It needs the ability to first *analyze* an incoming query and then *route* it to the correct destination or *construct* a more precise, structured query for retrieval. This section dives into the techniques that make this possible.

### Before we dive in

> There are 2 types of routing; logical and semantic 

![image.png](attachment:image.png)

We'll see both in detail

<a id='part3-1'></a>
## Logical Routing

Uses an LLM to reason through the prompt and classify it into predefined, discrete categories.

Routing is a classification problem. Given a user’s question, we need to classify it into one of several predefined categories. While traditional ML models can do this, we can leverage the powerful reasoning engine we already have: the LLM itself.

By providing the LLM with a clear schema (a set of possible categories), we can ask it to make the classification decision for us.

We’ll start by defining the “contract” for our LLM’s output using a Pydantic model. This schema explicitly tells the LLM the possible destinations for a query.

In [28]:
from typing import Literal
from pydantic import BaseModel, Field

# Define the data model for our router's output
class RouteQuery(BaseModel):
    """A data model to route a user query to the most relevant datasource."""

    # The 'datasource' field must be one of the three specified literal strings.
    # This enforces a strict set of choices for the LLM.
    datasource: Literal["python_docs", "js_docs", "golang_docs"] = Field(
        ...,  # The '...' indicates that this field is required.
        description="Given a user question, choose which datasource would be most relevant for answering their question.",
    )

With our schema defined, we can now build the router chain. We’ll use a prompt to give the LLM its instructions and then use the `.with_structured_output()` method to ensure its response perfectly matches our `RouteQuery` model.

In [29]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# Initialize Groq LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# Parser based on your Pydantic model
parser = JsonOutputParser(pydantic_object=RouteQuery)

# System prompt with explicit JSON instruction
system = """You are an expert at routing a user question to the appropriate data source.

Based on the programming language, decide the correct data source.

Return output strictly in JSON format:
{format_instructions}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
).partial(format_instructions=parser.get_format_instructions())

# Define router chain
router = prompt | llm | parser

Now, let’s test our router. We’ll pass it a question that is clearly about Python and inspect the output.

In [30]:
question = """Why doesn't the following code work:

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""

# Invoke the router and check the result
result = router.invoke({"question": question})

print(result)

{'datasource': 'python_docs'}


The output is an instance of our `RouteQuery` model, and the LLM has correctly identified `python_docs` as the appropriate datasource. This structured output is now something we can reliably use in our code to implement branching logic.

In [31]:
from langchain_core.runnables import RunnableLambda

def choose_route(result):
    """Determine downstream logic based on router output."""
    
    datasource = result.get("datasource", "").lower()

    if "python_docs" in datasource:
        return "chain for python_docs"
    elif "js_docs" in datasource:
        return "chain for js_docs"
    else:
        return "chain for golang_docs"

full_chain = router | RunnableLambda(choose_route)

final_destination = full_chain.invoke({"question": question})

print(final_destination)

chain for python_docs


Our switchboard correctly routed the Python-related query. This approach is incredibly powerful for building multi-source RAG systems.

<a id='part3-2'></a>
## Semantic Routing

Logical routing works perfectly when you have clearly defined categories. But what if you want to route based on the *style* or *domain* of a question? For example, you might want to answer physics questions with a serious, academic tone and math questions with a step-by-step, granular and learning approach. This is where **Semantic Routing** comes in.

> semantic routing uses vector embeddings to understand user intent. 

> Instead of classifying the query, we define multiple expert prompts.

We then embed the user’s query and each of our prompt templates, and use cosine similarity to find the prompt that is most semantically aligned with the query.

First, let’s define our two expert personas.

In [32]:
from langchain_core.prompts import PromptTemplate

# A prompt for a physics expert
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

# A prompt for a math expert
math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

Now, we’ll create the routing function that performs the embedding and similarity comparison.

In [33]:
from sklearn.metrics.pairwise import cosine_similarity
from langchain_community.embeddings import HuggingFaceEmbeddings

# Initialize the embedding model
embeddings = HuggingFaceEmbeddings()

# Store our templates and their embeddings for comparison
prompt_templates = [physics_template, math_template]
prompt_embeddings = embeddings.embed_documents(prompt_templates)

def prompt_router(input):
    """A function to route the input query to the most similar prompt template."""
    # 1. Embed the incoming user query
    query_embedding = embeddings.embed_query(input["query"])
    
    # 2. Compute the cosine similarity between the query and all prompt templates
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
    
    # 3. Find the index of the most similar prompt
    most_similar_index = similarity.argmax()
    
    # 4. Select the most similar prompt template
    chosen_prompt = prompt_templates[most_similar_index]
    
    print(f"DEBUG: Using {'MATH' if most_similar_index == 1 else 'PHYSICS'} template.")
    
    # 5. Return the chosen prompt object
    return PromptTemplate.from_template(chosen_prompt)

/var/folders/t1/wt965jp13hb3gls9w5dcy1bm0000gn/T/ipykernel_39802/3732559650.py:5: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5690.78it/s]


If in above cell it is returning a ModuleNotFoundError, with this import :
> from sklearn.metrics.pairwise import cosine_similarity

try using, this import :
> from langchain_core.utils.math import cosine_similarity

With the routing logic in place, we can build the full chain that dynamically selects the right expert for the job.

In [34]:
# The final chain that combines the router with the LLM
chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)  # Dynamically select the prompt
    | llm 
    | StrOutputParser()
)

# Ask a physics question
print(chain.invoke("What's a black hole"))

DEBUG: Using PHYSICS template.
A black hole is a region in space where the gravitational pull is so strong that nothing, including light, can escape. It's formed when a massive star collapses in on itself and its gravity becomes so strong that it warps the fabric of spacetime around it.

Imagine spacetime as a trampoline. If you place a heavy object, like a bowling ball, on the trampoline, it will warp and curve, creating a dent. That's similar to what a massive star does to spacetime. If the star is massive enough, the dent becomes so deep that it creates a boundary called the event horizon. Once something crosses the event horizon, it's trapped by the black hole's gravity and can't escape.

Black holes come in different sizes, ranging from small, stellar-mass black holes formed from star collapses, to supermassive black holes found at the centers of galaxies, with masses millions or even billions of times that of our sun.

That's a basic overview of black holes. Do you have any speci

This is Perfect! 

* As you notice, the router gave point-wise answer and used keywords like "imagine" and most importantly the **last** line, 
> "That's a basic overview of black holes. Do you have any specific questions about them or would you like me to elaborate on any aspect?"

The router correctly identified the question as physics-related and used the physics professor prompt, resulting in a concise and accurate answer. This technique is excellent for creating specialized agents that adapt their persona to the user’s needs.